# Fine-tuning d'un petit modèle de traduction FR → EN

Ce notebook sert **uniquement** à entraîner (affiner) un petit modèle de traduction français → anglais,
séparé de l'interface principale. L'interface de ton projet pourra ensuite proposer un onglet dédié
qui utilise ce modèle finetuné, tandis que le reste de l'appli continue d'utiliser le modèle pré-entraîné.

## 1. Installation des dépendances pour le fine-tuning
On installe les bibliothèques nécessaires pour le fine-tuning : `transformers`, `datasets`, etc.
Sur Colab, exécute cette cellule en premier.

In [ ]:
!pip install -q transformers datasets sentencepiece accelerate

## 2. Monter Google Drive (optionnel mais recommandé)
Si ton dataset et/ou l'endroit où tu veux sauvegarder le modèle sont sur Drive, monte-le ici.

In [ ]:
from google.colab import drive  # ignoré en local
drive.mount('/content/drive')

## 3. Charger ton dataset parallèle FR → EN
On part d'un fichier CSV avec au moins deux colonnes :
- `source` : phrase en français
- `target` : traduction en anglais

Exemple de chemin sur Drive : `/content/drive/MyDrive/datasets/fr_en_dataset.csv`.
Adapte la variable `csv_path` à ton cas.

In [ ]:
from datasets import load_dataset
import os

# À ADAPTER : chemin vers ton fichier CSV FR→EN
csv_path = '/content/drive/MyDrive/datasets/fr_en_dataset.csv'
assert os.path.exists(csv_path), f'Fichier introuvable : {csv_path}'

raw_datasets = load_dataset('csv', data_files={'train': csv_path})
raw_datasets

## 4. Préparer le modèle de base et le tokenizer
On utilise le modèle pré-entraîné `Helsinki-NLP/opus-mt-fr-en` comme point de départ.
On va seulement **affiner** ce modèle sur ton dataset, pas l'entraîner à partir de zéro.

In [ ]:
from transformers import MarianTokenizer, MarianMTModel

base_model_name = 'Helsinki-NLP/opus-mt-fr-en'
tokenizer = MarianTokenizer.from_pretrained(base_model_name)
model = MarianMTModel.from_pretrained(base_model_name)

print('Modèle et tokenizer chargés depuis', base_model_name)

## 5. Pré-traitement du dataset (tokenisation)
On transforme les colonnes `source` et `target` en entrées et labels pour le modèle.

In [ ]:
max_input_length = 128
max_target_length = 128

def preprocess_function(examples):
    inputs = examples['source']
    targets = examples['target']
    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True)

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(targets, max_length=max_target_length, truncation=True)
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)
tokenized_datasets

## 6. Configuration de l'entraînement (TrainingArguments + Trainer)
On définit quelques paramètres simples pour le fine-tuning.
Tu peux ajuster `num_train_epochs`, `per_device_train_batch_size`, etc., en fonction de la taille de ton dataset et des ressources Colab.

In [ ]:
from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

output_dir = './mt_fr_en_finetuned'
training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=8,
    num_train_epochs=3,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_steps=50,
    save_steps=500,
    save_total_limit=2,
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    data_collator=data_collator,
    tokenizer=tokenizer,
)

trainer

## 7. Lancer l'entraînement
Cette étape peut prendre du temps selon la taille de ton dataset et le runtime Colab (GPU recommandé).

In [ ]:
trainer.train()

## 8. Sauvegarder le modèle finetuné
On sauvegarde le modèle et le tokenizer dans un dossier (par exemple sur Drive).
Ensuite, l'interface de ton projet pourra charger ce dossier spécifique pour l'onglet "mon modèle".

In [ ]:
save_dir = '/content/drive/MyDrive/models/mt_fr_en_finetuned'  # à adapter si besoin
os.makedirs(save_dir, exist_ok=True)
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

print('Modèle finetuné sauvegardé dans :', save_dir)

## 9. Comment le relier à ton interface ?
- Dans `backend/translation.py`, tu pourras ajouter un mode spécial (par exemple une fonction `translate_text_custom`)
  qui charge ce modèle finetuné depuis `save_dir` au lieu de `Helsinki-NLP/opus-mt-fr-en`.
- Dans ton interface (onglet dédié), tu appelleras cette fonction spéciale pour utiliser **ton** modèle plutôt que le modèle général.

Je peux t'aider à modifier `backend/translation.py` pour ajouter ce support quand tu auras un modèle finetuné sauvegardé.